In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from scipy import sparse as sp

In [ ]:
merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df['transcript_id'] = merged_df['transcript_id'].str.split('.').str[0]


In [ ]:
meta = pd.read_csv('<RECIPE_PROJECT_ROOT>/data/GSE162060_HEK293Tscriboseq_meta.csv')
meta.head

In [ ]:
exp=pd.read_csv('<RECIPE_PROJECT_ROOT>/data/sc11619genes422cell.csv')
exp.head

In [ ]:
# 复制 DataFrame（注意加括号）
merged_df1 = exp.copy()

# 只选择数值型列
numeric_cols = merged_df1.select_dtypes(include='number').columns

# 对数值列求行平均
merged_df1["scribo"] = merged_df1[numeric_cols].mean(axis=1)
merged_df1.set_index("Unnamed: 0", inplace=True)

merged_df1

In [ ]:
# exp = exp[exp['transcript_id'].isin(exp_rich_filtered['transcript_id'])]
exp = exp.set_index('Unnamed: 0')
adata = ad.AnnData(X=exp)

# 使用 scanpy 的 normalize_total 方法进行标准化
sc.pp.normalize_total(adata)  # 规范化为每列总和为 1e6

# 转回 DataFrame 格式
exp_normalized = pd.DataFrame(
    adata.X, 
    index=adata.obs_names, 
    columns=adata.var_names
)
exp_normalized

In [ ]:
merged_df1["scribo"]

In [ ]:
# 复制 DataFrame（注意加括号）
merged_df = exp_normalized.copy()

# 只选择数值型列
numeric_cols = merged_df.select_dtypes(include='number').columns

# 对数值列求行平均
merged_df["scribo"] = merged_df1["scribo"]

# 保留 index 作为一列
merged_df["transcript_id"] = merged_df.index


In [ ]:
merged_df

In [ ]:
# pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_rich_pause.csv')
# pausing.shape

In [ ]:

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_rich_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscrich", "transcript_id"]

# 合并数据，缺失值填充为 0
merged_df2 = pd.merge(merged_df, pausing, on='transcript_id', how='left')
merged_df2['High_Pause_Countsscrich'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_leu6h_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscleu6h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing', '_new'))
merged_df2['High_Pause_Countsscleu6h'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_leu3h_pause.csv')
pausing.columns = ['protein_id', "High_Pause_Countsscleu3h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing', '_new'))
merged_df2['High_Pause_Countsscleu3h'].fillna(0, inplace=True)

pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_arg3h_pause.csv')

# 规范化列名
pausing.columns = ['protein_id', "High_Pause_Countsscarg3h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing1', '_new1'))
merged_df2['High_Pause_Countsscarg3h'].fillna(0, inplace=True)
pausing = pd.read_csv('<PAUSING_SOURCE_ROOT>/pause_scores/hek293t_arg6h_pause.csv')
pausing.columns = ['protein_id', "High_Pause_Countsscarg6h", "transcript_id"]
merged_df2 = pd.merge(merged_df2, pausing, on='transcript_id', how='left', suffixes=('_existing2', '_new2'))
merged_df2['High_Pause_Countsscarg6h'].fillna(0, inplace=True)


In [ ]:
from torch_geometric.utils import from_scipy_sparse_matrix
ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_pbulk11619.csv')

ppi_matrix.head()
all_sequence_outputsnew = np.load('./data/all_sequence_outputsnewbulk11619.npy')
ppi_matrix = sp.coo_matrix(ppi_matrix)
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

In [ ]:
merged_df.columns


In [ ]:
merged_df.to_csv('<RECIPE_PROJECT_ROOT>/data/single_cell/scribo_reference.csv')

从这开始

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad

# 读取数据
exp=pd.read_csv('<RECIPE_PROJECT_ROOT>/data/single_cell/scribo_reference.csv')

meta=pd.read_csv('brforepridictmeta_dataall.csv')

rich_cells = meta[meta['fraction'] == 'Rich']['cell_names'].tolist()

# 由于 cell_names 在 exp 中是列名，需要检查格式是否匹配
rich_cells_exp = [cell for cell in rich_cells if cell in exp.columns]

# 提取 Rich 细胞对应的表达矩阵
exp_rich = exp[['transcript_id'] + rich_cells_exp]

import pandas as pd

# 读取 exp_rich 数据（如果是 DataFrame 变量，直接用 exp_rich）
# exp_rich = pd.read_csv("your_file.csv")  # 如果是从 CSV 读取数据

# 删除所有列均为 0 的行
exp_rich_filtered = exp_rich.loc[~(exp_rich.iloc[:, 1:] == 0).all(axis=1)]

# 显示结果
exp_rich_filtered

# 如果需要保存到文件
# exp_rich_filtered.to_csv("filtered_exp_rich.csv", index=False)



In [ ]:
#分样本0.2
#12+12 Epoch: 3000, Train Loss: 0.130, Train R²: 0.866| Val Loss: 0.464, Val R²: 0.616|Test Loss: 0.482, Test R²: 0.515
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
from scipy import sparse as sp

import os
import random

def set_seed(seed=0):
    print('seed = {}'.format(seed))
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
# 设置随机种子
SEED = 12
set_seed(SEED)


class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        #print(f"pausescore shape: {pausescore.shape}")

        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore.view(-1, 1))), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z
# 训练与验证
device = torch.device("cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=3e-4)
criterion = nn.MSELoss()


neural_net.load_state_dict(torch.load('./models/single_cell_module_a_seed8_best.pt'))
#换成微调后的模型
neural_net = neural_net.to(device)  # 确保模型转移到了正确的设备
#bulk_unknown_standard_scaler_seed0.pt


In [ ]:
expk=pd.read_csv('<RECIPE_PROJECT_ROOT>/data/sc11619genes422cell.csv')
expk = expk.set_index('Unnamed: 0')
adata = ad.AnnData(X=expk)

# 使用 scanpy 的 normalize_total 方法进行标准化
sc.pp.normalize_total(adata)  # 规范化为每列总和为 1e6

# 转回 DataFrame 格式
exp_normalized = pd.DataFrame(
    adata.X, 
    index=adata.obs_names, 
    columns=adata.var_names
)
exp_normalized.head
#exp_normalized = np.nan_to_num(exp_normalized, nan=0)
exp_normalized = exp_normalized.fillna(0)


In [ ]:
has_nan = np.isnan(exp_normalized).any()
print("是否存在 NaN 值:", has_nan)


In [ ]:
merged_df2.shape


In [ ]:
# import numpy as np
# import torch
# from torch_geometric.data import Data
# from torch_geometric.utils import add_self_loops
# import pandas as pd

# # 初始化模型
# model = NeuralGraph()  # 初始化模型架构
# state_dict = torch.load('./models/single_cell_module_a_seed8_best.pt')
# model.load_state_dict(state_dict)
# model.eval()

# # 初始化空列表存储所有 z 和 y
# all_z = []
# all_y = []

# # # 加载 PPI 相关数据
# # #ppi_matrix = pd.read_csv('./data/ppi_ebi_string_ppi3ensp_lr_IntAct_corummatrix4p_p4431scribo.csv')
# # ppi_matrix = sp.coo_matrix(ppi_matrix)
# # edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# # # 加载基因序列数据
# # #all_sequence_outputsnew = np.load('./data/all_sequence_outputsnewclsscribo4431.npy')
# sequence_embedding = torch.tensor(all_sequence_outputsnew, dtype=torch.float32)  # [4431, 9216]

# # 提取不同组别的 pause_tensor
# pause_dict = {
#     "Rich": torch.tensor(merged_df2['High_Pause_Countsscrich'].to_numpy(), dtype=torch.float32).view(-1, 1),
#     "Leu3h": torch.tensor(merged_df2['High_Pause_Countsscleu3h'].to_numpy(), dtype=torch.float32).view(-1, 1),
#     "Leu6h": torch.tensor(merged_df2['High_Pause_Countsscleu6h'].to_numpy(), dtype=torch.float32).view(-1, 1),
#     "Arg3h": torch.tensor(merged_df2['High_Pause_Countsscarg3h'].to_numpy(), dtype=torch.float32).view(-1, 1),
#     "Arg6h": torch.tensor(merged_df2['High_Pause_Countsscarg6h'].to_numpy(), dtype=torch.float32).view(-1, 1),
# }

# # 遍历 exp_normalized_final 中的每个细胞（列）
# for column in exp_normalized.columns:  # 跳过 transcript_id
#     # 获取细胞所属组别
#     fraction = meta.loc[meta['cell_names'] == column, 'fraction'].values
#     if len(fraction) == 0:
#         continue  # 该细胞不在 meta 里，跳过
#     fraction = fraction[0]  # 提取组别名称

#     # 获取该细胞的基因表达数据
#     x_tensor = torch.tensor(exp_normalized[column].to_numpy(), dtype=torch.float32).view(-1, 1)  # [4431, 1]

#     # 选择对应的 pause_tensor
#     pause_tensor = pause_dict[fraction]  # 依据 fraction 选择 pause 数据

#     # 创建 `Data` 对象
#     data_obj = Data(
#         x=x_tensor,
#         edge_index=edge_index,
#         edge_weight=edge_weight,
#         seq=sequence_embedding,  # 使用 sequence_embedding
#         pause=pause_tensor  # 选择不同组别的 pause_tensor
#     )

#     # 添加自环
#     data_obj.edge_index, data_obj.edge_attr = add_self_loops(data_obj.edge_index, data_obj.edge_weight)

#     # 转移 `Data` 对象到模型所在设备
#     data_obj = data_obj.to('cpu')

#     # 获取 y 和 z
#     with torch.no_grad():
#         y, z = model(data_obj)

#     # 转换为 numpy 并存储
#     all_z.append(z.cpu().numpy())  # [4431, 32]
#     all_y.append(y.cpu().numpy())  # [4431, 1]

# # 转换为 numpy 数组
# all_z_array = np.array(all_z)  # [N, 4431, 32]
# all_y_array = np.array(all_y)  # [N, 4431, 1]

# print(f"Generated z shape: {all_z_array.shape}")
# print(f"Generated y shape: {all_y_array.shape}")

# # 保存文件
# np.save('data/all_z_array_0503test.npy', all_z_array)
# np.save('data/all_y_array_0503test.npy', all_y_array)


In [ ]:
exp_normalized.head

In [ ]:
# exp=exp_normalized

In [ ]:

import pandas as pd
import torch
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import numpy as np
exp=exp_normalized
def create_knn_graph(exp_data, n_neighbors=5, n_pcs=1):
    """
    构建表达量数据的KNN图，并返回边列表 (edge_index_knn)。
    
    Parameters:
    - exp_data: DataFrame, 表达量数据（细胞 x 基因矩阵）
    - n_neighbors: KNN中的邻居数量
    - n_pcs: PCA的主成分数量

    Returns:
    - edge_index_knn: PyTorch Tensor, 边列表，形状为 (2, num_edges)
    """

    # 先用PCA对表达量数据降维
    pca = PCA(n_components=n_pcs)
    exp_pca = pca.fit_transform(exp_data)

    print(f"Shape after PCA: {exp_pca.shape}")

    # 使用KNN算法，构建邻近图
    knn = NearestNeighbors(n_neighbors=n_neighbors + 1)  # +1 因为包含自身
    knn.fit(exp_pca)

    # 计算每个样本的K个最近邻
    distances, indices = knn.kneighbors(exp_pca)

    # 构建边列表
    edge_index_knn = []
    
    for node, neighbors in enumerate(indices):
        for neighbor in neighbors[1:]:  # 排除自身
            edge_index_knn.append([node, neighbor])

    # 将边列表转换为PyTorch的Tensor格式
    edge_index_knn = torch.tensor(edge_index_knn, dtype=torch.long).t().contiguous()

    print(f"Constructed edge_index_knn shape: {edge_index_knn.shape}")
    
    return edge_index_knn

# 载入表达量数据
# exp = exp_normalized_final
# 去掉第一列基因ID并转置矩阵
if isinstance(exp, np.ndarray):
    exp = pd.DataFrame(exp)  # 将 NumPy 数组转为 DataFrame

# 现在可以使用 .iloc
exp_values = exp.iloc[:, 0:].T.values
# 构建KNN图
edge_index_knn = create_knn_graph(exp_values, n_neighbors=5, n_pcs=1)

print(edge_index_knn)


In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from sklearn.neighbors import NearestNeighbors
import numpy as np
from torch.utils.data import random_split
# 设置随机种子，确保可重复性
seed = 0  # 你可以选择任何整数作为种子
torch.manual_seed(seed)
def create_knn_graph(exp_data, n_neighbors=3):
    """
    构建表达量数据的KNN图，并返回边列表 (edge_index_knn)。
    
    Parameters:
    - exp_data: NumPy array, 表达量数据（细胞 x 基因矩阵）
    - n_neighbors: KNN中的邻居数量

    Returns:
    - edge_index_knn: PyTorch Tensor, 边列表，形状为 (2, num_edges)
    """
    # 使用KNN算法，构建邻近图
    knn = NearestNeighbors(n_neighbors=n_neighbors + 1)  # +1 因为包含自身
    knn.fit(exp_data)

    # 计算每个样本的K个最近邻
    distances, indices = knn.kneighbors(exp_data)

    # 构建边列表
    edge_index_knn = []
    for node, neighbors in enumerate(indices):
        for neighbor in neighbors[1:]:  # 排除自身
            edge_index_knn.append([node, neighbor])

    # 将边列表转换为PyTorch的Tensor格式
    edge_index_knn = torch.tensor(edge_index_knn, dtype=torch.long).t().contiguous()

    return edge_index_knn

class GraphListDataset(Dataset):
    def __init__(self, graph_list, transform=None, pre_transform=None):
        super().__init__(None, transform, pre_transform)
        self.graph_list = graph_list

    def len(self):
        return len(self.graph_list)

    def get(self, idx):
        # 仅返回图数据对象
        return self.graph_list[idx]

# def create_knn_graphs_per_gene(exp, z_array, n_neighbors=3):
#     """
#     为每个基因构建 KNN 图的列表，使用每个基因的表达量构建细胞-细胞图，并将 z_array 中的特征作为节点特征。

#     Parameters:
#     - exp: 表达量数据 DataFrame，shape 为 (基因 x 细胞)
#     - z_array: 形状为 (215, 7132, 32) 的特征数组
#     - n_neighbors: KNN中的邻居数量

#     Returns:
#     - 一个包含所有基因的 KNN 图列表
#     """
#     all_graphs = []
    
#     for gene_idx in range(exp.shape[0]):  # exp.shape[0] 应该是 7132
#         # 获取当前基因的表达量数据 (细胞 x 1)
#         gene_exp_data = exp.iloc[gene_idx, :].values.reshape(-1, 1)
        
#         # 使用当前基因的表达量数据构建KNN图
#         edge_index_knn = create_knn_graph(gene_exp_data, n_neighbors=n_neighbors)
        
#         # 从 z_array 提取该基因的节点特征 (215, 32)
#         node_features = torch.tensor(z_array[:, gene_idx, :], dtype=torch.float)
        
#         # 创建 PyG 的图数据结构，并将 z_array 的特征作为 x
#         graph = Data(x=node_features, edge_index=edge_index_knn)
        
#         # 将图添加到列表
#         all_graphs.append(graph)
    
#     return all_graphs
import torch
from torch_geometric.data import Data
from sklearn.neighbors import NearestNeighbors
import numpy as np

def create_knn_graphs_per_cell(exp, z_array, meta, n_neighbors=3):
    """
    为每个细胞构建 KNN 图，图的节点是基因，边是基因之间的 KNN 关系。

    参数:
    - exp: (细胞数, 基因数) 的表达量矩阵
    - z_array: (细胞数, 基因数, 32) 的特征数组
    - meta: 细胞元数据 DataFrame
    - n_neighbors: KNN 近邻数量

    返回:
    - all_graphs: 包含所有细胞的 KNN 图列表
    """
    all_graphs = []
    num_cells, num_genes = exp.shape

    print(f"✅ 细胞数: {num_cells}, 基因数: {num_genes}")

    for cell_idx in range(num_cells):  
        # **获取当前细胞的基因表达数据**
        cell_exp_data = exp.iloc[cell_idx, :].values.reshape(-1, 1)  # (基因数, 1)

        # **构建 KNN 图 (基因之间的连接)**
        knn = NearestNeighbors(n_neighbors=n_neighbors + 1)  # +1 包含自身
        knn.fit(cell_exp_data)
        distances, indices = knn.kneighbors(cell_exp_data)

        # **构建 PyTorch Geometric 的边索引**
        edge_index_knn = []
        for node, neighbors in enumerate(indices):
            for neighbor in neighbors[1:]:  # 排除自身
                edge_index_knn.append([node, neighbor])
        edge_index_knn = torch.tensor(edge_index_knn, dtype=torch.long).t().contiguous()  # (2, num_edges)

        # **特征**
        node_features = torch.tensor(z_array[cell_idx, :, :], dtype=torch.float)  # (基因数, 32)

        # **确保 edge_index 不超界**
        num_nodes = node_features.shape[0]  # 预计为 num_genes
        if edge_index_knn.max().item() >= num_nodes:
            print(f"⚠️ 跳过细胞 {cell_idx}: Edge index 超界 ({edge_index_knn.max().item()} >= {num_nodes})")
            continue  # 跳过该细胞

        # **获取该细胞是否属于 Rich 组**
        is_rich = meta.iloc[cell_idx]['fraction'] == 'Rich'  # True / False
        rich_mask = torch.tensor(is_rich, dtype=torch.bool).repeat(num_genes)  # (基因数,)

        # **创建 PyG 数据对象**
        graph = Data(x=node_features, edge_index=edge_index_knn, rich_mask=rich_mask, gene_ids=torch.arange(num_genes))
        all_graphs.append(graph)

    return all_graphs




meta=pd.read_csv('brforepridictmeta_dataall.csv')
# 载入表达量数据
meta = meta.rename(columns={'Unnamed: 0': 'cell_names'})
# **检查 meta 列名**
print("Meta columns:", meta.columns)

# **如果 'cell_names' 列名重复，去重**
meta = meta.loc[:, ~meta.columns.duplicated()]

# **检查 cell_names 是否有重复**
print("Duplicate cell_names:", meta['cell_names'].duplicated().sum())  
meta = meta.drop_duplicates(subset=['cell_names'])

# 检查数据形状，确保行是基因，列是细胞
print(f"Data shape: {exp.shape}")  # 应该输出 (7132, 216)，其中一列是基因ID，215 列是细胞

# # 去掉第一列基因ID并转置矩阵 (确保行是基因，列是细胞)
# exp_values = exp.iloc[:, 1:]
#exp_values = exp.set_index('Unnamed: 0').iloc[:, 0:].T  # 确保 index 是细胞名
exp_values = exp.iloc[:, 0:].T  # 确保 index 是细胞名
# 加载 z_array，确保形状为 (215, 7132, 32)
z_array = np.load('data/all_z_array_0503test.npy')
print(f"z_array shape: {z_array.shape}")
# **加载 meta 数据**
meta = meta.set_index('cell_names').loc[exp_values.index]  # 对齐顺序


In [ ]:
z_array.shape

In [ ]:
exp.head

In [ ]:
# print("exp_values shape:", exp_values.shape)  # 应该是 (421, 4430)
# all_graphs = create_knn_graphs_per_cell(exp_values, z_array, meta, n_neighbors=3)


In [ ]:
exp_values.head

In [ ]:
print("exp_values shape:", exp_values.shape)

In [ ]:
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch
import numpy as np

def split_genes(num_genes, train_ratio=0.9, seed=42):
    """
    随机划分基因索引为训练集和测试集。
    """
    np.random.seed(seed)
    indices = np.random.permutation(num_genes)
    train_size = int(train_ratio * num_genes)
    train_genes = indices[:train_size]
    test_genes = indices[train_size:]
    return train_genes, test_genes

def create_knn_graphs_subset_genes(exp, z_array, meta, gene_indices, n_neighbors=3):
    """
    构建只包含部分基因的 KNN 图列表（每个细胞一张图，节点为基因）。
    """
    all_graphs = []
    num_cells = exp_values.shape[0]

    for cell_idx in range(num_cells):
        # 取该细胞在指定 gene_indices 下的表达量
        cell_exp_data = exp_values.iloc[cell_idx, gene_indices].values.reshape(-1, 1)

        # 构建 KNN 图
        knn = NearestNeighbors(n_neighbors=n_neighbors + 1)
        knn.fit(cell_exp_data)
        _, indices = knn.kneighbors(cell_exp_data)

        edge_index = []
        for node, neighbors in enumerate(indices):
            for neighbor in neighbors[1:]:
                edge_index.append([node, neighbor])
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        # 获取对应 gene 的 feature 向量
        node_features = torch.tensor(z_array[cell_idx, gene_indices, :], dtype=torch.float)

        # 标签（是否 Rich）
        is_rich = meta.iloc[cell_idx]['fraction'] == 'Rich'
        rich_mask = torch.tensor(is_rich, dtype=torch.bool).repeat(len(gene_indices))

        graph = Data(x=node_features, edge_index=edge_index, rich_mask=rich_mask, gene_ids=torch.tensor(gene_indices))
        all_graphs.append(graph)

    return all_graphs


In [ ]:
z_array.shape

In [ ]:
merged_df.shape

In [ ]:
cds_df=pd.read_csv("<PAUSING_SOURCE_ROOT>/cds_df38510.csv")
cds_df = cds_df.iloc[:,1:9]
cds_df['transcript_id'] = cds_df['transcript_id_x'].str.split('.').str[0]
exp=pd.read_csv("<PAUSING_SOURCE_ROOT>/data/sc11619genes422cell_normalized.csv")
# 将 cds_df 按照 exp 的基因列排序
sorted_cds_df = cds_df.set_index('transcript_id').reindex(exp['Unnamed: 0']).reset_index()

# 查看排序后的结果
sorted_cds_df.head()
sorted_cds_df.fillna(0, inplace=True)

merged_df2=sorted_cds_df
merged_df2['transcript_id'] = merged_df2['transcript_id_x'].str.split('.').str[0]
merged_df2.head

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

original_y = merged_df2["NC3"]
original_y_np = original_y.to_numpy()

total_gene_ids = np.arange(z_array.shape[1])  # 生成基因索引数组，从 0 到 11618

# 找出有 y 值的基因索引
valid_gene_ids = np.where(original_y != 0)[0]

# 找出没有 y 值的基因索引（11619 - 4431 = 7188）
invalid_gene_ids = np.setdiff1d(total_gene_ids, valid_gene_ids)

# 先将 valid_gene_ids 按 3:1:1 划分为 train / test / val
train_gene_ids, temp_gene_ids = train_test_split(valid_gene_ids, test_size=0.25, random_state=42)
test_gene_ids, val_gene_ids_with_y = train_test_split(temp_gene_ids, test_size=0.5, random_state=42)

# 把没有 y 的也归入 val
#val_gene_ids = np.concatenate([val_gene_ids_with_y, invalid_gene_ids])
val_gene_ids =val_gene_ids_with_y
print(f"Train genes (with y): {len(train_gene_ids)}")
print(f"Test genes (with y): {len(test_gene_ids)}")
print(f"Val genes (with y): {len(val_gene_ids_with_y)}")
print(f"Val genes total (with + without y): {len(val_gene_ids)}")


In [ ]:
# print("max gene index in val_gene_ids:", val_gene_ids.max())
# print("exp shape:", exp.shape)
# print("any invalid index:", np.any(val_gene_ids >= exp.shape[1]))
# print("any negative index:", np.any(val_gene_ids < 0))


In [ ]:
# gene 数量
num_genes = z_array.shape[1]
#exp = exp.T  # positional indexers are out-of-bounds
# exp.columns = exp.iloc[0]  # 把第一行作为列名
# exp = exp.drop(index=exp.index[0])  # 删除第一行
# exp = exp.reset_index(drop=True) 
# # 按基因划分训练/测试
# train_gene_indices, test_gene_indices = split_genes(num_genes, train_ratio=0.9)

# 创建图
train_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, train_gene_ids, n_neighbors=3)
test_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, test_gene_ids, n_neighbors=3)
val_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, val_gene_ids, n_neighbors=3)
# 创建 DataLoader
trainloader = DataLoader(train_graphs, batch_size=len(train_graphs), shuffle=False)
testloader = DataLoader(test_graphs, batch_size=len(test_graphs), shuffle=False)
valloader = DataLoader(val_graphs, batch_size=len(val_graphs), shuffle=False)
print(f"训练集基因数: {len(train_gene_ids)}, 图数: {len(train_graphs)}")
print(f"测试集基因数: {len(test_gene_ids)}, 图数: {len(test_graphs)}")

print(f"测试集基因数: {len(val_gene_ids)}, 图数: {len(val_graphs)}")

In [ ]:
# all_graphs = create_knn_graphs_per_cell(exp_values, z_array, meta, n_neighbors=3)

# # **划分训练集和测试集**
# dataset_size = len(all_graphs)
# train_size = int(0.9 * dataset_size)  # 90% 训练
# test_size = dataset_size - train_size  # 10% 测试

# train_dataset, test_dataset = random_split(all_graphs, [train_size, test_size])

# # **创建 DataLoader**
# trainloader = DataLoader(train_dataset, batch_size=train_size, drop_last=False, shuffle=False)
# testloader = DataLoader(test_dataset, batch_size=test_size, drop_last=False, shuffle=False)

# print(f"训练集: {len(train_dataset)}, 测试集: {len(test_dataset)}")


In [ ]:
# exp_values = exp.set_index('transcript_id').iloc[:, 1:].T  # 确保 index 是细胞名

# print("exp_values index:", exp_values.index[:5])  # 预计应该是基因 ID
# print("exp_values columns:", exp_values.columns[:5])  # 预计应该是细胞名
# print("meta columns:", meta.columns)


In [ ]:

# # **构建 KNN 图**

# # 为每个基因构建 KNN 图列表，每个图有 215 个节点（细胞），特征来自 z_array
# #all_graphs = create_knn_graphs_per_gene(exp_values, z_array, n_neighbors=3)

# # 检查生成的图数量
# print(f"Generated {len(all_graphs)} graphs.")  # 应该输出 7132 个图

# # 构建 GraphListDataset
# graph_dataset = GraphListDataset(all_graphs)

# # 数据集总长度
# dataset_size = len(graph_dataset)

# # 划分90%训练集和10%测试集
# train_size = int(0.9 * dataset_size)
# test_size = dataset_size - train_size
# # 设置随机种子
# torch.manual_seed(seed)
# # 随机划分训练集和测试集
# train_dataset, test_dataset = random_split(graph_dataset, [train_size, test_size])

# # 创建 trainloader 和 testloader
# trainloader = DataLoader(
#     train_dataset, 
#     batch_size=train_size,  # 使用整个训练集的大小
#     drop_last=False, 
#     shuffle=False  # 不需要 shuffle
# )


# testloader = DataLoader(
#     test_dataset, 
#     batch_size=test_size,  # 使用整个测试集的大小
#     drop_last=False, 
#     shuffle=False  # 测试集不需要 shuffle
# )


In [ ]:
# for data in trainloader:
#     print("Max edge index:", data.edge_index.max().item(), "Node count:", data.x.shape[0])
#     assert data.edge_index.max().item() < data.x.shape[0], "Edge index out of range!"


In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch_geometric.nn import SAGEConv
# from sklearn.metrics import r2_score
# import numpy as np
# import random
# import time

# # ✅ 设置随机种子
# def set_seed(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(12)
# # ✅ 数据预处理
# input_dim = 32
# output_dim = 16

# # ✅ 标签准备（你已经处理好）
# # merged_df2_filtered['NC3'] → CPM normalized values
# # original_y shape: (4431,)
# y_cpm_log2 = np.log2((merged_df2_filtered[['NC3']].values / np.median(merged_df2_filtered[['NC3']].values)) + 1)
# original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# # ✅ GraphSAGE 模型定义
# class GraphSAGE(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(GraphSAGE, self).__init__()
#         self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
#         self.mlp = nn.Sequential(
#             nn.Linear(output_dim, 16),
#             nn.GELU(),
#             nn.Linear(16, 1),
#         )

#     def forward(self, x, edge_index):
#         x = self.conv1(x, edge_index)
#         x = F.relu(x)
#         x = self.mlp(x)
#         x = F.relu(x)
#         return x.squeeze(-1)  # (num_nodes,)

# input_dim = 32  
# output_dim = 16  
# model = GraphSAGE(input_dim, output_dim).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# criterion = nn.MSELoss()

# def train(loader):
#     model.train()
#     total_loss_sum = 0.0
#     gene_count = 0

#     all_preds = []
#     all_labels = []

#     optimizer.zero_grad()
#     for batch in loader:
#         batch = batch.to(device)
#         y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
#         gene_ids = batch.gene_ids
#         rich_mask = batch.rich_mask.bool()
#         rich_preds = y_pred[rich_mask]
#         rich_gene_ids = gene_ids[rich_mask]

#         if rich_preds.numel() == 0:
#             continue

#         loss_sum = 0.0
#         losses = []
#         for gid in torch.unique(rich_gene_ids):
#             gid_mask = (rich_gene_ids == gid)
#             preds = rich_preds[gid_mask]
#             if preds.numel() == 0:
#                 continue
#             label = original_y[gid].repeat(preds.shape[0])
#             loss = F.mse_loss(preds, label, reduction='mean')
#             losses.append(loss)
#             total_loss_sum += loss.item()
#             gene_count += 1

#             all_preds.append(preds.detach().cpu().numpy())
#             all_labels.append(label.cpu().numpy())

#         # ✅ 一次性累加所有 loss 再反向传播
#         if len(losses) > 0:
#             total_loss = torch.stack(losses).mean()
#             total_loss.backward()

#     optimizer.step()
#     optimizer.zero_grad()

#     if gene_count > 0:
#         all_preds = np.concatenate(all_preds)
#         all_labels = np.concatenate(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#         avg_loss = total_loss_sum / gene_count
#     else:
#         avg_loss = 0.0
#         r2 = float('nan')

#     return avg_loss, r2



# def test(loader):
#     model.eval()

#     total_loss_sum = 0.0
#     gene_count = 0
#     all_preds = []
#     all_labels = []

#     with torch.no_grad():
#         for batch in loader:
#             batch = batch.to(device)
#             y_pred = model(batch.x, batch.edge_index)
#             gene_ids = batch.gene_ids
#             rich_mask = batch.rich_mask.bool()
#             rich_preds = y_pred[rich_mask]
#             rich_gene_ids = gene_ids[rich_mask]

#             if rich_preds.numel() == 0:
#                 continue

#             unique_gene_ids = torch.unique(rich_gene_ids)
#             for gid in unique_gene_ids:
#                 gid_mask = (rich_gene_ids == gid)
#                 preds = rich_preds[gid_mask]
#                 if preds.numel() == 0:
#                     continue
#                 label = original_y[gid].repeat(preds.shape[0])
#                 loss = F.mse_loss(preds, label, reduction='mean')

#                 total_loss_sum += loss.item()
#                 gene_count += 1

#                 all_preds.append(preds.cpu().numpy())
#                 all_labels.append(label.cpu().numpy())

#     if gene_count > 0:
#         all_preds = np.concatenate(all_preds)
#         all_labels = np.concatenate(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#         avg_loss = total_loss_sum / gene_count
#     else:
#         avg_loss = 0.0
#         r2 = float('nan')

#     return avg_loss, r2
# num_epochs = 391
# for epoch in range(1, num_epochs + 1):
#     train_loss, train_r2 = train(trainloader)
#     test_loss, test_r2 = test(testloader)

#     print(f"Epoch {epoch:03d} | "
#           f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
#           f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ 数据预处理
input_dim = 32
output_dim = 16



# ✅ 标签准备（你已经处理好）
# merged_df2_filtered['NC3'] → CPM normalized values
# original_y shape: (4431,)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# ✅ 直接提取数据
def get_batch(loader):
    return next(iter(loader)).to(device)

# ✅ 训练函数
def train(loader):
    model.train()
    optimizer.zero_grad()

    batch = get_batch(loader)  # 获取唯一的批次数据
    y_pred = model(batch.x, batch.edge_index)  # 所有基因节点的预测值
    
    rich_mask = batch.rich_mask.bool()
    gene_ids = batch.gene_ids

    if rich_mask.sum() == 0:
        return 0.0, float('nan')

    # ✅ 计算 rich 组中每个基因的预测均值
    rich_preds = y_pred[rich_mask]
    rich_gene_ids = gene_ids[rich_mask]

    unique_gene_ids = torch.unique(rich_gene_ids)

    losses = []
    all_preds = []
    all_labels = []
    total_loss_sum = 0.0
    gene_count = 0

    for gid in unique_gene_ids:
        gid_mask = (rich_gene_ids == gid)
        preds = rich_preds[gid_mask]

        if preds.numel() == 0:
            continue

        # ✅ 计算 rich 组的预测均值
        mean_pred = preds.mean()

        # ✅ 获取真实值
        label = original_y[gid]

        # ✅ MSE 损失计算
        loss = F.mse_loss(mean_pred, label)
        losses.append(loss)
        total_loss_sum += loss.item()
        gene_count += 1

        all_preds.append(mean_pred.detach().cpu().numpy())
        all_labels.append(label.cpu().numpy())

    # ✅ 一次性反向传播
    if len(losses) > 0:
        total_loss = torch.stack(losses).mean()
        total_loss.backward()
        optimizer.step()

    # ✅ 计算 R²
    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2

# ✅ 测试函数
def test(loader):
    model.eval()

    batch = get_batch(loader)  # 获取唯一的批次数据
    y_pred = model(batch.x, batch.edge_index)
    
    rich_mask = batch.rich_mask.bool()
    gene_ids = batch.gene_ids

    if rich_mask.sum() == 0:
        return 0.0, float('nan')

    rich_preds = y_pred[rich_mask]
    rich_gene_ids = gene_ids[rich_mask]

    unique_gene_ids = torch.unique(rich_gene_ids)
    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for gid in unique_gene_ids:
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() == 0:
                continue

            mean_pred = preds.mean()
            label = original_y[gid]
            
            loss = F.mse_loss(mean_pred, label)
            total_loss_sum += loss.item()
            gene_count += 1

            all_preds.append(mean_pred.cpu().numpy())
            all_labels.append(label.cpu().numpy())

    # ✅ 计算 R²
    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2
# ✅ 测试/验证函数，修改为只计算有标签的基因
# def val(loader, val_gene_ids_with_y):
#     model.eval()

#     batch = get_batch(loader)  # 获取唯一的批次数据
#     y_pred = model(batch.x, batch.edge_index)
    
#     rich_mask = batch.rich_mask.bool()
#     gene_ids = batch.gene_ids

#     if rich_mask.sum() == 0:
#         return 0.0, float('nan')

#     rich_preds = y_pred[rich_mask]
#     rich_gene_ids = gene_ids[rich_mask]

#     unique_gene_ids = torch.unique(rich_gene_ids)
#     total_loss_sum = 0.0
#     gene_count = 0
#     all_preds = []
#     all_labels = []

#     with torch.no_grad():
#         for gid in unique_gene_ids:
#             gid_mask = (rich_gene_ids == gid)
#             preds = rich_preds[gid_mask]

#             if preds.numel() == 0:
#                 continue

#             # 只计算有标签的基因
#             if gid not in val_gene_ids_with_y:
#                 continue

#             mean_pred = preds.mean()
#             label = original_y[gid]
            
#             loss = F.mse_loss(mean_pred, label)
#             total_loss_sum += loss.item()
#             gene_count += 1

#             all_preds.append(mean_pred.cpu().numpy())
#             all_labels.append(label.cpu().numpy())

#     # ✅ 计算 R²
#     if gene_count > 0:
#         all_preds = np.array(all_preds)
#         all_labels = np.array(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#         avg_loss = total_loss_sum / gene_count
#     else:
#         avg_loss = 0.0
#         r2 = float('nan')

#     return avg_loss, r2

# ✅ 训练循环与早停实现
best_test_r2 = float('-inf')
patience_counter = 0
patience = 50
best_train_loss = None
best_val_loss = None
best_test_loss = None
best_val_r2 = float('-inf')

num_epochs = 391
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)
    val_loss, val_r2 = test(valloader)  # 测试集用于验证

    # 早停检查
    if test_r2 > best_test_r2:  # 使用 test集的 R² 作为早停指标
        best_test_r2 = test_r2
        best_train_loss = train_loss
        best_val_loss = val_loss
        best_test_loss = test_loss
        best_val_r2 = val_r2  # 保存验证集的 R²
        patience_counter = 0

       # torch.save(model.state_dict(), './250504scribobest_model.pth')  # 保存最优模型
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Epoch: {epoch:03d}, Train Loss: {train_loss:.3f}, Train R²: {train_r2:.3f}| "
          f"Val Loss: {val_loss:.3f}, Val R²: {val_r2:.3f}| "
          f"Test Loss: {best_test_loss:.3f}, Test R²: {best_test_r2:.3f}")


In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch_geometric.nn import SAGEConv
# from sklearn.metrics import r2_score
# import numpy as np
# import random
# import time

# # ✅ 设置随机种子
# def set_seed(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(12)

# # ✅ 数据预处理
# input_dim = 32
# output_dim = 16
# model = GraphSAGE(input_dim, output_dim).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# # ✅ 标签准备（你已经处理好）
# # merged_df2_filtered['NC3'] → CPM normalized values
# # original_y shape: (4431,)
# y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df2[['NC3']].values)) + 1)
# original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# # ✅ GraphSAGE 模型定义
# class GraphSAGE(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(GraphSAGE, self).__init__()
#         self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
#         self.mlp = nn.Sequential(
#             nn.Linear(output_dim, 16),
#             nn.GELU(),
#             nn.Linear(16, 1),
#         )

#     def forward(self, x, edge_index):
#         x = self.conv1(x, edge_index)
#         x = F.relu(x)
#         x = self.mlp(x)
#         x = F.relu(x)
#         return x.squeeze(-1)

# input_dim = 32  
# output_dim = 16  
# model = GraphSAGE(input_dim, output_dim).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# criterion = nn.MSELoss()

# # ✅ 直接提取数据
# def get_batch(loader):
#     return next(iter(loader)).to(device)

# # ✅ 训练函数
# def train(loader):
#     model.train()
#     optimizer.zero_grad()

#     batch = get_batch(loader)  # 获取唯一的批次数据
#     y_pred = model(batch.x, batch.edge_index)  # 所有基因节点的预测值
    
#     rich_mask = batch.rich_mask.bool()
#     gene_ids = batch.gene_ids

#     if rich_mask.sum() == 0:
#         return 0.0, float('nan')

#     # ✅ 计算 rich 组中每个基因的预测均值
#     rich_preds = y_pred[rich_mask]
#     rich_gene_ids = gene_ids[rich_mask]

#     unique_gene_ids = torch.unique(rich_gene_ids)

#     losses = []
#     all_preds = []
#     all_labels = []
#     total_loss_sum = 0.0
#     gene_count = 0

#     for gid in unique_gene_ids:
#         gid_mask = (rich_gene_ids == gid)
#         preds = rich_preds[gid_mask]

#         if preds.numel() == 0:
#             continue

#         # ✅ 计算 rich 组的预测均值
#         mean_pred = preds.mean()

#         # ✅ 获取真实值
#         label = original_y[gid]

#         # ✅ MSE 损失计算
#         loss = F.mse_loss(mean_pred, label)
#         losses.append(loss)
#         total_loss_sum += loss.item()
#         gene_count += 1

#         all_preds.append(mean_pred.detach().cpu().numpy())
#         all_labels.append(label.cpu().numpy())

#     # ✅ 一次性反向传播
#     if len(losses) > 0:
#         total_loss = torch.stack(losses).mean()
#         total_loss.backward()
#         optimizer.step()

#     # ✅ 计算 R²
#     if gene_count > 0:
#         all_preds = np.array(all_preds)
#         all_labels = np.array(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#         avg_loss = total_loss_sum / gene_count
#     else:
#         avg_loss = 0.0
#         r2 = float('nan')

#     return avg_loss, r2

# # ✅ 测试函数
# def test(loader):
#     model.eval()

#     batch = get_batch(loader)  # 获取唯一的批次数据
#     y_pred = model(batch.x, batch.edge_index)
    
#     rich_mask = batch.rich_mask.bool()
#     gene_ids = batch.gene_ids

#     if rich_mask.sum() == 0:
#         return 0.0, float('nan')

#     rich_preds = y_pred[rich_mask]
#     rich_gene_ids = gene_ids[rich_mask]

#     unique_gene_ids = torch.unique(rich_gene_ids)
#     total_loss_sum = 0.0
#     gene_count = 0
#     all_preds = []
#     all_labels = []

#     with torch.no_grad():
#         for gid in unique_gene_ids:
#             gid_mask = (rich_gene_ids == gid)
#             preds = rich_preds[gid_mask]

#             if preds.numel() == 0:
#                 continue

#             mean_pred = preds.mean()
#             label = original_y[gid]
            
#             loss = F.mse_loss(mean_pred, label)
#             total_loss_sum += loss.item()
#             gene_count += 1

#             all_preds.append(mean_pred.cpu().numpy())
#             all_labels.append(label.cpu().numpy())

#     # ✅ 计算 R²
#     if gene_count > 0:
#         all_preds = np.array(all_preds)
#         all_labels = np.array(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#         avg_loss = total_loss_sum / gene_count
#     else:
#         avg_loss = 0.0
#         r2 = float('nan')

#     return avg_loss, r2

# # ✅ 训练循环
# num_epochs = 391
# for epoch in range(1, num_epochs + 1):
#     train_loss, train_r2 = train(trainloader)
#     test_loss, test_r2 = test(testloader)

#     print(f"Epoch {epoch:03d} | "
#           f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
#           f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")


In [ ]:
# torch.save(model, './models/single_cell_graph_seed12_best.pt')
# torch.save(model.state_dict(), './models/single_cell_graph_seed12_state.pt')

In [ ]:
from itertools import chain

model.eval()

# 初始化预测值与计数器
y_all_pred = torch.zeros_like(original_y)         # shape: (4431,)
gene_rich_counts = torch.zeros_like(original_y)   # shape: (4431,)

with torch.no_grad():
    for batch in chain(trainloader, testloader, valloader):  # ✅ 加入 valloader
        batch = batch.to(device)

        # 模型预测
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        rich_mask = batch.rich_mask.bool()
        gene_ids = batch.gene_ids

        # 提取 rich 节点的预测和 gene_id
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        # 聚合每个基因的预测值
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() > 0:
                y_all_pred[gid] += preds.sum()
                gene_rich_counts[gid] += preds.shape[0]

# 平均化（防止除以 0）
y_all_pred = y_all_pred / gene_rich_counts.clamp(min=1)


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 转 numpy
original_y_np = original_y.cpu().numpy()
y_all_pred_np = y_all_pred.cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_heldout.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import DataLoader
from itertools import chain

# ✅ 使用你全数据的 gene_indices
all_gene_indices = np.arange(z_array.shape[1])  # [0, 1, ..., 4430]

# ✅ 构造完整图（基于所有基因）
all_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, all_gene_indices, n_neighbors=3)

# ✅ 创建 DataLoader
graph_loader = DataLoader(all_graphs, batch_size=len(all_graphs), shuffle=False)

# ✅ 初始化预测矩阵
num_cells = len(all_graphs)
num_genes = len(all_gene_indices)
predicted_matrix = np.zeros((num_cells, num_genes))

# ✅ 推理填入预测矩阵
model.eval()
with torch.no_grad():
    for batch in graph_loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)

        for i in range(num_cells):
            node_mask = (batch.batch == i)
            preds = y_pred[node_mask].cpu().numpy().squeeze()
            predicted_matrix[i, :] = preds

# ✅ 转成 DataFrame，恢复行列名
predicted_df = pd.DataFrame(
    predicted_matrix,
    index=exp_values.index,      # 细胞名作为行名
    columns=exp_values.columns   # 基因名作为列名
)

# ✅ 保存为 CSV
predicted_df.to_csv("predicted_expression_421_11619_250504.csv")


print("✅ predicted_expression_421_11619_250504.csv")
predicted_df.head

In [ ]:
# 统计每一列是否全为 0，返回布尔 Series
all_zero_columns = (predicted_df == 0).all(axis=0)

# 计算全为 0 的列数量
num_all_zero_columns = all_zero_columns.sum()

print(f"全为 0 的列数: {num_all_zero_columns}")


In [ ]:
from itertools import chain  # ✅ 用于合并 DataLoader

model.eval()

# 初始化预测值与计数器
y_all_pred = torch.zeros_like(original_y)         # shape: (4431,)
gene_rich_counts = torch.zeros_like(original_y)   # shape: (4431,)

with torch.no_grad():
    for batch in chain(trainloader, testloader):  # ✅ 正确合并两个 DataLoader
        batch = batch.to(device)

        # 模型预测
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        rich_mask = batch.rich_mask.bool()         # 挑出 rich 细胞内的节点
        gene_ids = batch.gene_ids                  # 每个节点的 gene ID

        # 只选出 rich 节点的预测和对应 gene_id
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        # 聚合每个基因在 rich 中的预测值
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() > 0:
                y_all_pred[gid] += preds.sum()
                gene_rich_counts[gid] += preds.shape[0]

# 平均值（防止除以 0）
y_all_pred = y_all_pred / gene_rich_counts.clamp(min=1)



In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 转 numpy
original_y_np = original_y.cpu().numpy()
y_all_pred_np = y_all_pred.cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_pink.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#D6A47A'
hist_color = '#BFD2DF'

# ✅ 转 numpy
original_y_np = original_y.cpu().numpy()
y_all_pred_np = y_all_pred.cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_orange.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#AAAFCB'
hist_color = '#BFD2DF'

# ✅ 转 numpy
original_y_np = original_y.cpu().numpy()
y_all_pred_np = y_all_pred.cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_blue.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import DataLoader
from itertools import chain

# ✅ 使用你全数据的 gene_indices
all_gene_indices = np.arange(z_array.shape[1])  # [0, 1, ..., 4430]

# ✅ 构造完整图（基于所有基因）
all_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, all_gene_indices, n_neighbors=3)

# ✅ 创建 DataLoader
graph_loader = DataLoader(all_graphs, batch_size=len(all_graphs), shuffle=False)

# ✅ 初始化预测矩阵
num_cells = len(all_graphs)
num_genes = len(all_gene_indices)
predicted_matrix = np.zeros((num_cells, num_genes))

# ✅ 推理填入预测矩阵
model.eval()
with torch.no_grad():
    for batch in graph_loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)

        for i in range(num_cells):
            node_mask = (batch.batch == i)
            preds = y_pred[node_mask].cpu().numpy().squeeze()
            predicted_matrix[i, :] = preds

# ✅ 转成 DataFrame，恢复行列名
predicted_df = pd.DataFrame(
    predicted_matrix,
    index=exp_values.index,      # 细胞名作为行名
    columns=exp_values.columns   # 基因名作为列名
)

# ✅ 保存为 CSV
predicted_df.to_csv("./outputs/single_cell_predictions.csv")


print("✅ ./outputs/single_cell_predictions.csv")
predicted_df.head


# 绘图

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ 数据预处理
input_dim = 32
output_dim = 16


merged_df = pd.read_csv('./data/24077132kdncmergedf.csv')
merged_df['transcript_id'] = merged_df['transcript_id'].str.split('.').str[0]

# ✅ 标签准备（你已经处理好）
# merged_df2_filtered['NC3'] → CPM normalized values
# original_y shape: (4431,)
y_cpm_log2 = np.log2((merged_df2[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


In [ ]:
# 加载训练好的模型
model.load_state_dict(torch.load('./250504scribobest_model.pth'))
model.eval()
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl
# 初始化预测值与计数器
y_test_pred = torch.zeros_like(original_y)
gene_test_counts = torch.zeros_like(original_y)

with torch.no_grad():
    for batch in testloader:
        batch = batch.to(device)

        y_pred = model(batch.x, batch.edge_index)
        rich_mask = batch.rich_mask.bool()
        gene_ids = batch.gene_ids

        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() > 0:
                y_test_pred[gid] += preds.sum()
                gene_test_counts[gid] += preds.shape[0]

# 计算平均值
y_test_pred = y_test_pred / gene_test_counts.clamp(min=1)

# ✅ 只取 test_gene_ids 中有标签的基因
test_gene_ids_tensor = torch.tensor(test_gene_ids, device=device)
valid_test_ids = test_gene_ids_tensor[original_y[test_gene_ids_tensor] != 0]

original_y_np = original_y[valid_test_ids].cpu().numpy()
y_test_pred_np = y_test_pred[valid_test_ids].cpu().numpy()

# ✅ 相关分析
correlation = np.corrcoef(original_y_np, y_test_pred_np)[0, 1]
result = pg.corr(original_y_np, y_test_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ 格式化 P 值
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 绘图
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)
ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# 主图
ax_scatter.scatter(original_y_np, y_test_pred_np, alpha=0.6, color=scatter_color)
max_val = max(original_y_np.max(), y_test_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')
sns.regplot(x=original_y_np, y=y_test_pred_np, scatter=False, color='black', ax=ax_scatter)
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_test_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)
ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_test.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import pingouin as pg
import numpy as np
import torch

# ✅ 设置矢量字体以确保 PDF 可编辑
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'  # 可改为 'Arial'，确保系统已安装

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 模型加载
model.load_state_dict(torch.load('./250504scribobest_model.pth'))
model.eval()

# ✅ 初始化预测值与计数器
y_train_pred = torch.zeros_like(original_y)
gene_train_counts = torch.zeros_like(original_y)

with torch.no_grad():
    for batch in trainloader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)

        rich_mask = batch.rich_mask.bool()
        gene_ids = batch.gene_ids
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]
            if preds.numel() > 0:
                y_train_pred[gid] += preds.sum()
                gene_train_counts[gid] += preds.shape[0]

# ✅ 平均化预测值
y_train_pred = y_train_pred / gene_train_counts.clamp(min=1)

# ✅ 提取 train 中有标签的部分
train_gene_ids_tensor = torch.tensor(train_gene_ids, device=device)
valid_train_ids = train_gene_ids_tensor[original_y[train_gene_ids_tensor] != 0]

original_y_np = original_y[valid_train_ids].cpu().numpy()
y_train_pred_np = y_train_pred[valid_train_ids].cpu().numpy()

# ✅ Pearson 相关分析
correlation = np.corrcoef(original_y_np, y_train_pred_np)[0, 1]
result = pg.corr(original_y_np, y_train_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ 格式化 P 值
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 绘图
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)
ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 主图
ax_scatter.scatter(original_y_np, y_train_pred_np, alpha=0.6, color=scatter_color)
max_val = max(original_y_np.max(), y_train_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')
sns.regplot(x=original_y_np, y=y_train_pred_np, scatter=False, color='black', ax=ax_scatter)

ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_train_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)
ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# ✅ 保存为 PDF，确保文字可编辑
plt.savefig("./outputs/figures/single_cell_mask_prediction_train.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import pingouin as pg
import numpy as np
import torch
from itertools import chain

# ✅ 设置字体使 PDF 可编辑
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'DejaVu Sans'  # 确保字体可导出

# ✅ 加载模型
model.load_state_dict(torch.load('./250504scribobest_model.pth'))
model.eval()

# ✅ 初始化结果
y_pred_all = torch.zeros_like(original_y)
count_all = torch.zeros_like(original_y)

# ✅ 合并所有子集：train + test + val
all_loader = chain(trainloader, testloader, valloader)

# ✅ 遍历所有 batch 聚合预测
with torch.no_grad():
    for batch in all_loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)

        rich_mask = batch.rich_mask.bool()
        gene_ids = batch.gene_ids

        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]
            if preds.numel() > 0:
                y_pred_all[gid] += preds.sum()
                count_all[gid] += preds.shape[0]

# ✅ 平均每个基因的预测值
y_pred_all = y_pred_all / count_all.clamp(min=1)

# ✅ 仅保留有标签的基因
valid_ids = (original_y != 0).nonzero(as_tuple=True)[0]
y_true_np = original_y[valid_ids].cpu().numpy()
y_pred_np = y_pred_all[valid_ids].cpu().numpy()

# ✅ Pearson 相关与 P 值
correlation = np.corrcoef(y_true_np, y_pred_np)[0, 1]
result = pg.corr(y_true_np, y_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ 格式化 P 值
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    sci = f"{p_value:.2e}"
    base, exp = sci.split("e")
    p_value_tex = f"{float(base):.2f} \\times 10^{{{int(exp)}}}"

# ✅ 开始绘图
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)
ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🎨 配色
scatter_color = '#C7B0C4'
hist_color = '#BFD2DF'

# ✅ 散点图
ax_scatter.scatter(y_true_np, y_pred_np, alpha=0.6, color=scatter_color)
max_val = max(y_true_np.max(), y_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')
sns.regplot(x=y_true_np, y=y_pred_np, scatter=False, color='black', ax=ax_scatter)

ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# ✅ 边缘直方图
ax_histx.hist(y_true_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)
ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# ✅ 保存 PDF
plt.savefig("./outputs/figures/single_cell_mask_prediction_all.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ 数据预处理
merged_df = merged_df2
original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)

# 使用 Log2 CPM 归一化
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)

# 转换为 PyTorch Tensor
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)
# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)  # (num_nodes,)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = nn.MSELoss()

def train(loader):
    model.train()
    optimizer.zero_grad()

    total_loss_sum = 0.0
    gene_count = 0

    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)  # 所有基因节点的预测值
        rich_mask = batch.rich_mask.bool()
        gene_ids = batch.gene_ids

        if rich_mask.sum() == 0:
            continue

        # ✅ 计算 rich 组中每个基因的预测均值
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        unique_gene_ids = torch.unique(rich_gene_ids)
        losses = []

        for gid in unique_gene_ids:
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() == 0:
                continue

            # ✅ 计算 rich 组的预测均值
            mean_pred = preds.mean()

            # ✅ 获取真实值
            label = original_y[gid]

            # ✅ MSE 损失计算
            loss = F.mse_loss(mean_pred, label)
            losses.append(loss)
            total_loss_sum += loss.item()
            gene_count += 1

            all_preds.append(mean_pred.detach().cpu().numpy())
            all_labels.append(label.detach().cpu().numpy())


        # ✅ 一次性反向传播
        if len(losses) > 0:
            total_loss = torch.stack(losses).mean()
            total_loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    # ✅ 计算 R²
    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2
def test(loader):
    model.eval()

    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            y_pred = model(batch.x, batch.edge_index)
            rich_mask = batch.rich_mask.bool()
            gene_ids = batch.gene_ids

            if rich_mask.sum() == 0:
                continue

            rich_preds = y_pred[rich_mask]
            rich_gene_ids = gene_ids[rich_mask]

            unique_gene_ids = torch.unique(rich_gene_ids)

            for gid in unique_gene_ids:
                gid_mask = (rich_gene_ids == gid)
                preds = rich_preds[gid_mask]

                if preds.numel() == 0:
                    continue

                mean_pred = preds.mean()
                label = original_y[gid]
                
                loss = F.mse_loss(mean_pred, label)
                total_loss_sum += loss.item()
                gene_count += 1

                all_preds.append(mean_pred.detach().cpu().numpy())
                all_labels.append(label.detach().cpu().numpy())


    # ✅ 计算 R²
    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2

num_epochs = 2000
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    print(f"Epoch {epoch:03d} | "
          f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
          f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)
merged_df=merged_df2
original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)  # (7132,)
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)  # (num_nodes,)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
def train(loader):
    model.train()
    total_loss_sum = 0.0
    gene_count = 0

    all_preds = []
    all_labels = []

    optimizer.zero_grad()
    for batch in loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        gene_ids = batch.gene_ids
        rich_mask = batch.rich_mask.bool()
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        if rich_preds.numel() == 0:
            continue

        loss_sum = 0.0
        losses = []
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]
            
            if preds.numel() == 0:
                continue

            # ✅ 计算预测均值
            mean_pred = preds.mean()

            # ✅ 获取真实值
            label = original_y[gid]

            # ✅ 计算 MSE Loss
            loss = F.mse_loss(mean_pred, label)
            losses.append(loss)
            total_loss_sum += loss.item()
            gene_count += 1

            # ✅ 修复 detach 问题
            all_preds.append(mean_pred.detach().cpu().numpy())
            all_labels.append(label.cpu().numpy())

        # ✅ 一次性累加所有 loss 再反向传播
        if len(losses) > 0:
            total_loss = torch.stack(losses).mean()
            total_loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2


def test(loader):
    model.eval()

    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            y_pred = model(batch.x, batch.edge_index)
            gene_ids = batch.gene_ids
            rich_mask = batch.rich_mask.bool()
            rich_preds = y_pred[rich_mask]
            rich_gene_ids = gene_ids[rich_mask]

            if rich_preds.numel() == 0:
                continue

            unique_gene_ids = torch.unique(rich_gene_ids)
            for gid in unique_gene_ids:
                gid_mask = (rich_gene_ids == gid)
                preds = rich_preds[gid_mask]
                
                if preds.numel() == 0:
                    continue

                # ✅ 计算预测均值
                mean_pred = preds.mean()

                # ✅ 获取真实值
                label = original_y[gid]

                # ✅ 计算 MSE Loss
                loss = F.mse_loss(mean_pred, label)

                total_loss_sum += loss.item()
                gene_count += 1

                # ✅ 修复 detach 问题
                all_preds.append(mean_pred.detach().cpu().numpy())
                all_labels.append(label.cpu().numpy())

    if gene_count > 0:
        all_preds = np.array(all_preds)
        all_labels = np.array(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2

num_epochs = 2000
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    print(f"Epoch {epoch:03d} | "
          f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
          f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)  # (num_nodes,)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

def train(loader):
    model.train()
    total_loss_sum = 0.0
    gene_count = 0

    all_preds = []
    all_labels = []

    optimizer.zero_grad()
    for batch in loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        gene_ids = batch.gene_ids
        rich_mask = batch.rich_mask.bool()
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        if rich_preds.numel() == 0:
            continue

        loss_sum = 0.0
        losses = []
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]
            if preds.numel() == 0:
                continue
            label = original_y[gid].repeat(preds.shape[0])
            loss = F.mse_loss(preds, label, reduction='mean')
            losses.append(loss)
            total_loss_sum += loss.item()
            gene_count += 1

            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(label.cpu().numpy())

        # ✅ 一次性累加所有 loss 再反向传播
        if len(losses) > 0:
            total_loss = torch.stack(losses).mean()
            total_loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2



def test(loader):
    model.eval()

    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            y_pred = model(batch.x, batch.edge_index)
            gene_ids = batch.gene_ids
            rich_mask = batch.rich_mask.bool()
            rich_preds = y_pred[rich_mask]
            rich_gene_ids = gene_ids[rich_mask]

            if rich_preds.numel() == 0:
                continue

            unique_gene_ids = torch.unique(rich_gene_ids)
            for gid in unique_gene_ids:
                gid_mask = (rich_gene_ids == gid)
                preds = rich_preds[gid_mask]
                if preds.numel() == 0:
                    continue
                label = original_y[gid].repeat(preds.shape[0])
                loss = F.mse_loss(preds, label, reduction='mean')

                total_loss_sum += loss.item()
                gene_count += 1

                all_preds.append(preds.cpu().numpy())
                all_labels.append(label.cpu().numpy())

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2
num_epochs = 2000
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    print(f"Epoch {epoch:03d} | "
          f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
          f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)  # (num_nodes,)

input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

def train(loader):
    model.train()
    total_loss_sum = 0.0
    gene_count = 0

    all_preds = []
    all_labels = []

    optimizer.zero_grad()
    for batch in loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        gene_ids = batch.gene_ids
        rich_mask = batch.rich_mask.bool()
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        if rich_preds.numel() == 0:
            continue

        loss_sum = 0.0
        losses = []
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]
            if preds.numel() == 0:
                continue
            label = original_y[gid].repeat(preds.shape[0])
            loss = F.mse_loss(preds, label, reduction='mean')
            losses.append(loss)
            total_loss_sum += loss.item()
            gene_count += 1

            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(label.cpu().numpy())

        # ✅ 一次性累加所有 loss 再反向传播
        if len(losses) > 0:
            total_loss = torch.stack(losses).mean()
            total_loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2



def test(loader):
    model.eval()

    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            y_pred = model(batch.x, batch.edge_index)
            gene_ids = batch.gene_ids
            rich_mask = batch.rich_mask.bool()
            rich_preds = y_pred[rich_mask]
            rich_gene_ids = gene_ids[rich_mask]

            if rich_preds.numel() == 0:
                continue

            unique_gene_ids = torch.unique(rich_gene_ids)
            for gid in unique_gene_ids:
                gid_mask = (rich_gene_ids == gid)
                preds = rich_preds[gid_mask]
                if preds.numel() == 0:
                    continue
                label = original_y[gid].repeat(preds.shape[0])
                loss = F.mse_loss(preds, label, reduction='mean')

                total_loss_sum += loss.item()
                gene_count += 1

                all_preds.append(preds.cpu().numpy())
                all_labels.append(label.cpu().numpy())

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2
num_epochs = 2000
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    print(f"Epoch {epoch:03d} | "
          f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
          f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f}")


In [ ]:
torch.save(model, './models/single_cell_graph_alternative_best.pt')
torch.save(model.state_dict(), './models/single_cell_graph_alternative_state.pt')

In [ ]:
from itertools import chain  # ✅ 用于合并 DataLoader

model.eval()

# 初始化预测值与计数器
y_all_pred = torch.zeros_like(original_y)         # shape: (4431,)
gene_rich_counts = torch.zeros_like(original_y)   # shape: (4431,)

with torch.no_grad():
    for batch in chain(trainloader, testloader):  # ✅ 正确合并两个 DataLoader
        batch = batch.to(device)

        # 模型预测
        y_pred = model(batch.x, batch.edge_index)  # (num_nodes,)
        rich_mask = batch.rich_mask.bool()         # 挑出 rich 细胞内的节点
        gene_ids = batch.gene_ids                  # 每个节点的 gene ID

        # 只选出 rich 节点的预测和对应 gene_id
        rich_preds = y_pred[rich_mask]
        rich_gene_ids = gene_ids[rich_mask]

        # 聚合每个基因在 rich 中的预测值
        for gid in torch.unique(rich_gene_ids):
            gid_mask = (rich_gene_ids == gid)
            preds = rich_preds[gid_mask]

            if preds.numel() > 0:
                y_all_pred[gid] += preds.sum()
                gene_rich_counts[gid] += preds.shape[0]

# 平均值（防止除以 0）
y_all_pred = y_all_pred / gene_rich_counts.clamp(min=1)



In [ ]:
# ✅ 导入依赖
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 支持
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#8d91c0'
hist_color = '#b8acb9'

# ✅ 转 numpy
original_y_np = original_y.cpu().numpy()
y_all_pred_np = y_all_pred.cpu().numpy()

# ✅ Pearson 相关 + p 值
correlation = np.corrcoef(original_y_np, y_all_pred_np)[0, 1]
result = pg.corr(original_y_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ P 值格式化
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签与标注
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predicted protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_alternative.pdf", format="pdf", bbox_inches="tight")

# 📈 展示
plt.show()


In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import DataLoader
from itertools import chain

# ✅ 使用你全数据的 gene_indices
all_gene_indices = np.arange(z_array.shape[1])  # [0, 1, ..., 4430]

# ✅ 构造完整图（基于所有基因）
all_graphs = create_knn_graphs_subset_genes(exp_values, z_array, meta, all_gene_indices, n_neighbors=3)

# ✅ 创建 DataLoader
graph_loader = DataLoader(all_graphs, batch_size=len(all_graphs), shuffle=False)

# ✅ 初始化预测矩阵
num_cells = len(all_graphs)
num_genes = len(all_gene_indices)
predicted_matrix = np.zeros((num_cells, num_genes))

# ✅ 推理填入预测矩阵
model.eval()
with torch.no_grad():
    for batch in graph_loader:
        batch = batch.to(device)
        y_pred = model(batch.x, batch.edge_index)

        for i in range(num_cells):
            node_mask = (batch.batch == i)
            preds = y_pred[node_mask].cpu().numpy().squeeze()
            predicted_matrix[i, :] = preds

# ✅ 转成 DataFrame，恢复行列名
predicted_df = pd.DataFrame(
    predicted_matrix,
    index=exp_values.index,      # 细胞名作为行名
    columns=exp_values.columns   # 基因名作为列名
)

# ✅ 保存为 CSV
predicted_df.to_csv("./outputs/single_cell_predictions_alternative.csv")


print("✅ ./outputs/single_cell_predictions_alternative.csv")
predicted_df.head

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random
import time

# ✅ 设置随机种子
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)

# ✅ GraphSAGE 模型定义
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        x = F.relu(x)
        return x.squeeze(-1)  # (num_nodes,)

# ✅ 初始化
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
input_dim = 32
output_dim = 16
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# ✅ 标签准备（你已经处理好）
# merged_df2_filtered['NC3'] → CPM normalized values
# original_y shape: (4431,)
y_cpm_log2 = np.log2((merged_df2_filtered[['NC3']].values / np.median(merged_df2_filtered[['NC3']].values)) + 1)
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# ✅ 训练函数
def train(loader):
    model.train()
    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    for data in loader:
        data = data.to(device)
        y_pred = model(data.x, data.edge_index)
        batch = data.batch
        gene_ids = data.gene_ids
        rich_mask = data.rich_mask.bool()
        num_graphs = gene_ids.shape[0]

        optimizer.zero_grad()

        for i in range(num_graphs):
            node_mask = (batch == i)
            graph_rich_mask = rich_mask[node_mask]
            graph_preds = y_pred[node_mask]

            if graph_rich_mask.sum() == 0:
                continue

            rich_preds = graph_preds[graph_rich_mask]
            gene_id = gene_ids[i]
            label = original_y[gene_id]
            label_vec = label.repeat(rich_preds.shape[0])

            loss = F.mse_loss(rich_preds, label_vec, reduction='mean')
            loss.backward()
            total_loss_sum += loss.item()
            gene_count += 1

            all_preds.append(rich_preds.detach().cpu().numpy())
            all_labels.append(label_vec.cpu().numpy())

        optimizer.step()

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2

# ✅ 测试函数
def test(loader):
    model.eval()
    total_loss_sum = 0.0
    gene_count = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            y_pred = model(data.x, data.edge_index)
            batch = data.batch
            gene_ids = data.gene_ids
            rich_mask = data.rich_mask.bool()
            num_graphs = gene_ids.shape[0]

            for i in range(num_graphs):
                node_mask = (batch == i)
                graph_rich_mask = rich_mask[node_mask]
                graph_preds = y_pred[node_mask]

                if graph_rich_mask.sum() == 0:
                    continue

                rich_preds = graph_preds[graph_rich_mask]
                gene_id = gene_ids[i]
                label = original_y[gene_id]
                label_vec = label.repeat(rich_preds.shape[0])

                loss = F.mse_loss(rich_preds, label_vec, reduction='mean')
                total_loss_sum += loss.item()
                gene_count += 1

                all_preds.append(rich_preds.cpu().numpy())
                all_labels.append(label_vec.cpu().numpy())

    if gene_count > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
        avg_loss = total_loss_sum / gene_count
    else:
        avg_loss = 0.0
        r2 = float('nan')

    return avg_loss, r2
num_epochs = 821

for epoch in range(1, num_epochs + 1):
    start_time = time.time()

    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    elapsed = time.time() - start_time
    print(f"Epoch {epoch:03d} | "
          f"Train Loss: {train_loss:.6f}, R²: {train_r2:.4f} | "
          f"Test Loss: {test_loss:.6f}, R²: {test_r2:.4f} | "
          f"Time: {elapsed:.2f}s")


In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch_geometric.nn import SAGEConv
# from sklearn.metrics import r2_score
# import numpy as np
# import random

# # ✅ **设置随机种子**
# def set_seed(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(12)  # 设定种子

# # ✅ **定义 GraphSAGE 模型**
# class GraphSAGE(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(GraphSAGE, self).__init__()
#         self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
#         self.mlp = nn.Sequential(
#             nn.Linear(output_dim, 16),
#             nn.GELU(),
#             nn.Linear(16, 1),
#         )

#     def forward(self, x, edge_index):
#         x = self.conv1(x, edge_index)
#         x = F.relu(x)
#         x = self.mlp(x)
#         return x  # (num_nodes, 1)

# # ✅ **初始化模型**
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# input_dim = 32  
# output_dim = 16  
# model = GraphSAGE(input_dim, output_dim).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# criterion = nn.MSELoss()

# # ✅ **数据准备**
# merged_df = merged_df2_filtered
# original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)  # (4431,)

# # **处理 Log 归一化**
# y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)
# original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# # ✅ **获取训练集和测试集索引**
# train_indices = train_dataset.indices if hasattr(train_dataset, 'indices') else list(range(len(train_dataset)))
# test_indices = test_dataset.indices if hasattr(test_dataset, 'indices') else list(range(len(test_dataset)))

# # ✅ **训练函数**
# def train(loader):
#     model.train()
#     optimizer.zero_grad()
#     total_loss = 0
#     num_batches = 0

#     all_preds = []
#     all_labels = []

#     for data in loader:
#         data = data.to(device)
#         y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

#         # **获取 Rich 细胞的 mask**
#         rich_mask = data.rich_mask.view(-1)  # (基因数,)

#         # **确保 Rich 细胞存在**
#         if rich_mask.sum() == 0:
#             continue  # 没有 Rich 细胞，跳过

#         # **提取 Rich 细胞的预测值**
#         rich_preds = y_pred[rich_mask].squeeze(-1)  # 🚀 `squeeze` 让形状匹配

#         # **获取基因索引并计算 Loss**
#         gene_indices = data.gene_ids
#         rich_labels = original_y[gene_indices][rich_mask]

#         # **检查维度匹配**
#         if rich_preds.shape != rich_labels.shape:
#             print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
#             continue

#         loss = criterion(rich_preds, rich_labels)
#         loss.backward()
#         optimizer.step()

#         total_loss += loss.item()
#         num_batches += 1

#         # **收集预测值和真实值**
#         all_preds.append(rich_preds.cpu().detach().numpy())
#         all_labels.append(rich_labels.cpu().detach().numpy())

#     # **计算 R²**
#     if len(all_preds) > 0 and len(all_labels) > 0:
#         all_preds = np.concatenate(all_preds)
#         all_labels = np.concatenate(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#     else:
#         r2 = float('nan')

#     return total_loss / num_batches if num_batches > 0 else 0.0, r2  # 避免除以 0

# def test(loader):
#     model.eval()
#     total_loss = 0
#     num_batches = 0

#     all_preds = []
#     all_labels = []

#     with torch.no_grad():
#         for data in loader:
#             data = data.to(device)
#             y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

#             # **获取 Rich 细胞的 mask**
#             rich_mask = data.rich_mask.view(-1)  # (基因数,)

#             # **确保 Rich 细胞存在**
#             if rich_mask.sum() == 0:
#                 continue  # 没有 Rich 细胞，跳过

#             # **提取 Rich 细胞的预测值**
#             rich_preds = y_pred[rich_mask].squeeze(-1)  # 🚀 `squeeze` 让形状匹配

#             # **获取基因索引并计算 Loss**
#             gene_indices = data.gene_ids
#             rich_labels = original_y[gene_indices][rich_mask]

#             # **检查维度匹配**
#             if rich_preds.shape != rich_labels.shape:
#                 print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
#                 continue

#             loss = criterion(rich_preds, rich_labels)
#             total_loss += loss.item()
#             num_batches += 1

#             # **收集预测值和真实值**
#             all_preds.append(rich_preds.cpu().numpy())
#             all_labels.append(rich_labels.cpu().numpy())

#     # **计算 R²**
#     if len(all_preds) > 0 and len(all_labels) > 0:
#         all_preds = np.concatenate(all_preds)
#         all_labels = np.concatenate(all_labels)
#         r2 = r2_score(all_labels, all_preds)
#     else:
#         r2 = float('nan')

#     return total_loss / num_batches if num_batches > 0 else 0.0, r2  # 避免除以 0

# num_epochs = 821
# for epoch in range(1, num_epochs + 1):
#     train_loss, train_r2 = train(trainloader)
#     test_loss, test_r2 = test(testloader)

#     print(f"Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Train R²: {train_r2:.4f}, Test Loss: {test_loss:.4f}, Test R²: {test_r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random

# ✅ **设置随机种子**
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(12)  # 设定种子

# ✅ **定义 GraphSAGE 模型**
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        
        return x  # (num_nodes, 1)

# ✅ **初始化模型**
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
input_dim = 32  
output_dim = 16  
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# ✅ **数据准备**
merged_df = merged_df2_filtered
original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)  # (4431,)

# **处理 Log 归一化**
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)
original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# ✅ **获取训练集和测试集索引**
train_indices = train_dataset.indices if hasattr(train_dataset, 'indices') else list(range(len(train_dataset)))
test_indices = test_dataset.indices if hasattr(test_dataset, 'indices') else list(range(len(test_dataset)))

# ✅ **训练函数**
def train(loader):
    model.train()
    optimizer.zero_grad()
    total_loss = 0
    num_batches = 0

    all_preds = []
    all_labels = []

    for data in loader:
        data = data.to(device)
        y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

        # **获取 Rich 细胞的 mask**
        rich_mask = data.rich_mask.view(-1)  # (基因数,)

        # **确保 Rich 细胞存在**
        if rich_mask.sum() == 0:
            continue  # 没有 Rich 细胞，跳过

        # **提取 Rich 细胞的预测值**
        rich_preds = y_pred[rich_mask].squeeze(-1)  # 🚀 `squeeze` 让形状匹配

        # **获取基因索引并计算 Loss**
        gene_indices = data.gene_ids
        rich_labels = original_y[gene_indices][rich_mask]

        # **检查维度匹配**
        if rich_preds.shape != rich_labels.shape:
            print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
            continue

        loss = criterion(rich_preds, rich_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        # **收集预测值和真实值**
        all_preds.append(rich_preds.cpu().detach().numpy())
        all_labels.append(rich_labels.cpu().detach().numpy())

    # **计算 R²**
    if len(all_preds) > 0 and len(all_labels) > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
    else:
        r2 = float('nan')

    return total_loss / num_batches if num_batches > 0 else 0.0, r2  # 避免除以 0

def test(loader):
    model.eval()
    total_loss = 0
    num_batches = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

            # **获取 Rich 细胞的 mask**
            rich_mask = data.rich_mask.view(-1)  # (基因数,)

            # **确保 Rich 细胞存在**
            if rich_mask.sum() == 0:
                continue  # 没有 Rich 细胞，跳过

            # **提取 Rich 细胞的预测值**
            rich_preds = y_pred[rich_mask].squeeze(-1)  # 🚀 `squeeze` 让形状匹配

            # **获取基因索引并计算 Loss**
            gene_indices = data.gene_ids
            rich_labels = original_y[gene_indices][rich_mask]

            # **检查维度匹配**
            if rich_preds.shape != rich_labels.shape:
                print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
                continue

            loss = criterion(rich_preds, rich_labels)
            total_loss += loss.item()
            num_batches += 1

            # **收集预测值和真实值**
            all_preds.append(rich_preds.cpu().numpy())
            all_labels.append(rich_labels.cpu().numpy())

    # **计算 R²**
    if len(all_preds) > 0 and len(all_labels) > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
    else:
        r2 = float('nan')

    return total_loss / num_batches if num_batches > 0 else 0.0, r2  # 避免除以 0

num_epochs = 821
for epoch in range(1, num_epochs + 1):
    train_loss, train_r2 = train(trainloader)
    test_loss, test_r2 = test(testloader)

    print(f"Epoch {epoch:03d}, Train Loss: {train_loss:.4f}, Train R²: {train_r2:.4f}, Test Loss: {test_loss:.4f}, Test R²: {test_r2:.4f}")


In [ ]:
torch.save(model, './models/single_cell_graph_baseline.pt')
torch.save(model.state_dict(), './models/single_cell_graph_baseline_state.pt')

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch_geometric.nn import SAGEConv
# from sklearn.metrics import r2_score
# import numpy as np
# import random

# # ✅ **设置随机种子**
# def set_seed(seed):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(seed)
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# set_seed(12)  # 设定种子

# # ✅ **定义 GraphSAGE 模型**
# class GraphSAGE(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(GraphSAGE, self).__init__()
#         self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
#         self.mlp = nn.Sequential(
#             nn.Linear(output_dim, 16),
#             nn.GELU(),
#             nn.Linear(16, 1),
#         )

#     def forward(self, x, edge_index):
#         x = self.conv1(x, edge_index)
#         x = F.relu(x)
#         x = self.mlp(x)
#         return x  # (num_nodes, 1)

# # ✅ **初始化模型**
# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# input_dim = 32  
# output_dim = 16  
# model = GraphSAGE(input_dim, output_dim).to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
# criterion = nn.MSELoss()

# # ✅ **数据准备**
# merged_df = merged_df2_filtered
# original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)  # (4431,)

# # **处理 Log 归一化**
# y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values)) + 1)
# original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)

# # ✅ **获取训练集和测试集索引**
# train_indices = train_dataset.indices if hasattr(train_dataset, 'indices') else list(range(len(train_dataset)))
# test_indices = test_dataset.indices if hasattr(test_dataset, 'indices') else list(range(len(test_dataset)))


In [ ]:
model.load_state_dict(torch.load('./models/single_cell_graph_baseline_state.pt', map_location=device))


model = model.to(device)  # 确保模型转移到了正确的设备
#bulk_unknown_standard_scaler_seed0.pt


In [ ]:
# # 初始化空的列表，用于保存所有的目标值
# all_y = []
# # merged_df['NC1'] 是目标值（original_y）
# original_y = torch.tensor(merged_df['NC1'].to_numpy(), dtype=torch.float32).to(device)  # (7132,)
# original_y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1).to(device)


# # 遍历训练集的所有数据
# for data in trainloader:
#     # 获取当前批次的目标值 y (假设目标值存在 data.y 中)
#     y_batch = original_y[train_dataset.indices]  # 取出训练集对应的原始目标值
#     all_y.append(y_batch)

# # 遍历测试集的所有数据
# for data in testloader:
#     # 获取当前批次的目标值 y (假设目标值存在 data.y 中)
#     y_batch = original_y[test_dataset.indices]  # 取出测试集对应的原始目标值
#     all_y.append(y_batch)

# # 将所有批次的目标值拼接成一个完整的向量
# original_y2 = torch.cat(all_y, dim=0)

# # 输出 original_y2 的长度，应该与 7132 一致
# print(f"Original_y2 length: {len(original_y2)}")


In [ ]:
with torch.no_grad():
    model.eval()
    predicted_y = []
    corresponding_labels = []

    for loader in [trainloader, testloader]:
        for data in loader:
            data = data.to(device)
            y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

            rich_mask = data.rich_mask.view(-1)

            if rich_mask.sum() == 0:
                continue

            # 提取 rich 区域的预测值和真实值
            rich_preds = y_pred[rich_mask].squeeze(-1)
            gene_indices = data.gene_ids
            rich_labels = original_y[gene_indices][rich_mask]

            if rich_preds.shape != rich_labels.shape:
                print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
                continue

            predicted_y.append(rich_preds.cpu())
            corresponding_labels.append(rich_labels.cpu())

    # 拼接所有图的预测值与标签
    y_all_pred = torch.cat(predicted_y).squeeze()
    y_all_true = torch.cat(corresponding_labels).squeeze()

    if len(y_all_pred) != len(y_all_true):
        print(f"❌ Error: The length of y_all_pred ({len(y_all_pred)}) does not match y_all_true ({len(y_all_true)}).")
    else:
        full_r2 = r2_score(y_all_true.numpy(), y_all_pred.numpy())
        print(f'📊 Full Dataset R²: {full_r2:.4f}')



In [ ]:
# Modify the previous code to remove the x-axis from the top histogram and ensure no duplicate y-axis on the right histogram.

# Necessary imports
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import matplotlib as mpl
mpl.rcParams['text.usetex'] = False
import pingouin as pg
# Define colors
scatter_color = '#8d91c0'  # Color for the scatter points
hist_color = '#b8acb9'     # Color for the histograms

# Assuming original_y2 and y_all_pred are tensors, convert them to NumPy arrays after moving to CPU
original_y2_np = y_all_true.numpy()
y_all_pred_np = y_all_pred.numpy()
correlation = np.corrcoef(original_y2_np, y_all_pred_np)[0, 1]


# Calculate correlation between the true and predicted values
correlation = np.corrcoef(original_y2_np, y_all_pred_np)[0, 1]

# Create the scatter plot with marginal histograms, refined
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

# Scatter plot and histograms layout
ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# Scatter plot with #8d91c0 color
ax_scatter.scatter(original_y2_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# Plot the diagonal reference line
max_val = max(original_y2_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# Using seaborn's regplot to add the regression line with a gray distribution band
sns.regplot(x=original_y2_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# Set labels and title
ax_scatter.set_xlabel('True', fontsize=12)
ax_scatter.set_ylabel('Predict', fontsize=12)
ax_scatter.text(0.05, 0.9, f'correlation: {correlation:.4f}', transform=ax_scatter.transAxes)

# Histograms with #b8acb9 color
ax_histx.hist(original_y2_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# Remove extra y-axis from the right histogram to avoid duplicate with scatter plot
ax_histy.yaxis.set_visible(False)

# Keep only the vertical axis for the top histogram
ax_histx.set_ylabel("Frequency")  # Show the axis for the top histogram
ax_histx.xaxis.set_visible(False)  # Remove the x-axis from the top histogram
import matplotlib as mpl

# 启用 LaTeX 渲染和嵌入字体
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42  # 嵌入可编辑字体
mpl.rcParams['ps.fonttype'] = 42  # 确保 PS 文件的字体也是可编辑的

# 绘图部分的代码保持不变
plt.savefig("./outputs/figures/single_cell_mask_prediction_summary.pdf", format="pdf", bbox_inches="tight")


# Display the final plot with regression line and diagonal line
plt.show()


In [ ]:
# 📦 导入
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import torch
import pingouin as pg
import matplotlib as mpl

# ✅ LaTeX 渲染和字体嵌入
mpl.rcParams['text.usetex'] = True
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 🎨 配色
scatter_color = '#8d91c0'
hist_color = '#b8acb9'

# ✅ 转换张量为 NumPy
original_y2_np = y_all_true.numpy()
y_all_pred_np = y_all_pred.numpy()

# ✅ 计算相关性（保留 np.corrcoef）
correlation = np.corrcoef(original_y2_np, y_all_pred_np)[0, 1]

# ✅ 使用 pingouin 计算 p 值
result = pg.corr(original_y2_np, y_all_pred_np, method='pearson')
p_value = result['p-val'][0]

# ✅ 格式化 P 值
if p_value < 1e-300:
    p_value_tex = r"< 1 \times 10^{-300}"
else:
    p_value_sci = f"{p_value:.2e}"
    p_base, p_exp = p_value_sci.split("e")
    p_value_tex = f"{float(p_base):.2f} \\times 10^{{{int(p_exp)}}}"

# ✅ 图形与布局
fig = plt.figure(figsize=(6, 6))
gs = GridSpec(4, 4)

ax_scatter = fig.add_subplot(gs[1:4, 0:3])
ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)

# 🔵 散点图
ax_scatter.scatter(original_y2_np, y_all_pred_np, alpha=0.6, color=scatter_color)

# ➕ 对角线参考线
max_val = max(original_y2_np.max(), y_all_pred_np.max())
ax_scatter.plot([0, max_val], [0, max_val], ls='--', color='black')

# 🔁 回归线
sns.regplot(x=original_y2_np, y=y_all_pred_np, scatter=False, color='black', ax=ax_scatter)

# 🏷️ 标签
ax_scatter.set_xlabel('Real protein expression', fontsize=12)
ax_scatter.set_ylabel('Predict protein expression', fontsize=12)
ax_scatter.text(0.05, 0.9, f'Correlation: {correlation:.4f}', transform=ax_scatter.transAxes, fontsize=10)
ax_scatter.text(0.05, 0.85, f'P ${p_value_tex}$', transform=ax_scatter.transAxes, fontsize=10, style='italic')

# 📊 边缘直方图
ax_histx.hist(original_y2_np, bins=40, color=hist_color, alpha=0.6)
ax_histy.hist(y_all_pred_np, bins=40, orientation='horizontal', color=hist_color, alpha=0.6)

# 去重轴
ax_histy.yaxis.set_visible(False)
ax_histx.set_ylabel("Frequency")
ax_histx.xaxis.set_visible(False)

# 💾 保存
plt.savefig("./outputs/figures/single_cell_mask_prediction_summary.pdf", format="pdf", bbox_inches="tight")

# 展示
plt.show()


In [ ]:
len(original_y2_np)




In [ ]:
 with torch.no_grad():
        for data in loader:
            data = data.to(device)
            y_pred = model(data.x, data.edge_index)  # (num_nodes, 1)

            # **获取 Rich 细胞的 mask**
            rich_mask = data.rich_mask.view(-1)  # (基因数,)

            # **确保 Rich 细胞存在**
            if rich_mask.sum() == 0:
                continue  # 没有 Rich 细胞，跳过

            # **提取 Rich 细胞的预测值**
            rich_preds = y_pred[rich_mask].squeeze(-1)  # 🚀 `squeeze` 让形状匹配

            # **获取基因索引并计算 Loss**
            gene_indices = data.gene_ids
            rich_labels = original_y[gene_indices][rich_mask]

            # **检查维度匹配**
            if rich_preds.shape != rich_labels.shape:
                print(f"❌ Shape mismatch: rich_preds={rich_preds.shape}, rich_labels={rich_labels.shape}")
                continue

            loss = criterion(rich_preds, rich_labels)
            total_loss += loss.item()
            num_batches += 1

            # **收集预测值和真实值**
            all_preds.append(rich_preds.cpu().numpy())
            all_labels.append(rich_labels.cpu().numpy())

    # **计算 R²**
    if len(all_preds) > 0 and len(all_labels) > 0:
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        r2 = r2_score(all_labels, all_preds)
    else:
        r2 = float('nan')

    return total_loss / num_batches if num_batches > 0 else 0.0, r2  # 避免除以 0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import r2_score
import numpy as np
import random

def set_seed(seed):
    random.seed(seed)  # 为 Python 内置的随机数生成器设置种子
    np.random.seed(seed)  # 为 NumPy 随机数生成器设置种子
    torch.manual_seed(seed)  # 为 CPU 上的 PyTorch 随机数生成器设置种子
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)  # 为当前 GPU 设置种子
        torch.cuda.manual_seed_all(seed)  # 为所有可用的 GPU 设置种子
    torch.backends.cudnn.deterministic = True  # 确保每次结果一致
    torch.backends.cudnn.benchmark = False  # 禁用 CUDNN 自动优化

# 设置随机种子
set_seed(12)  # 你可以将 42 替换为你需要的任意种子值

# 假设模型类为 GraphSAGE
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(input_dim, output_dim, aggr='mean')
        self.mlp = nn.Sequential(
            nn.Linear(output_dim, 16),
            nn.GELU(),
            nn.Linear(16, 1),  # 最终输出维度为 1
            #nn.Sigmoid() 
        )

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.mlp(x)
        out = F.relu(x)
        return out  # 输出为 (num_nodes, 1)

# 设备配置
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
input_dim = 32  # 输入特征的维度
output_dim = 16  # GraphSAGE 输出的隐藏特征维度
model = GraphSAGE(input_dim, output_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
#model = GraphSAGE(input_dim, output_dim).to(device)
model.load_state_dict(torch.load('./models/single_cell_graph_baseline_state.pt', map_location=device))
model = model.to(device)  # 确保模型转移到了正确的设备
model.eval()  


In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from src.utils import set_seed
from sklearn.metrics import r2_score
import torch.nn.functional as F
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from src.utils import set_seed
from sklearn.metrics import r2_score
import torch.nn.functional as F

from src.utils import set_seed
from sklearn.metrics import r2_score
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

dataloader = DataLoader(
    all_graphs, 
    batch_size=7132,  # 使用整个训练集的大小
    drop_last=False, 
    shuffle=False  # 不需要 shuffle
)

# 准备一个空的 (7132, 215) 矩阵，用来存放预测结果
predicted_matrix = np.zeros((421,4431))

# 使用模型进行预测
with torch.no_grad():
    for data in trainloader:
        data = data.to(device)
        y_pred = model(data.x, data.edge_index)  # 获取预测结果
        
        # 遍历每个图，提取节点的预测值
        batch_size = data.ptr.shape[0] - 1
        for i in range(batch_size):
            start, end = data.ptr[i], data.ptr[i + 1]
            gene_idx = i  # 由于每个图对应一个基因，索引即为图的索引
            predicted_matrix[gene_idx, :] = y_pred[start:end].cpu().numpy().squeeze()  # 将预测值填入矩阵中

    for data in testloader:
        data = data.to(device)
        y_pred = model(data.x, data.edge_index)
        
        # 同样遍历每个图并填充预测矩阵
        batch_size = data.ptr.shape[0] - 1
        for i in range(batch_size):
            start, end = data.ptr[i], data.ptr[i + 1]
            gene_idx = i + len(trainloader.dataset)  # 测试集基因索引从训练集后面开始
            predicted_matrix[gene_idx, :] = y_pred[start:end].cpu().numpy().squeeze()

# 保存预测矩阵为CSV文件
predicted_df = pd.DataFrame(predicted_matrix)



In [ ]:
exp_values

In [ ]:
predicted_df

In [ ]:
cell_ids = exp_values.index  # 细胞 ID

gene_ids = exp_values.columns  # 基因 ID

predicted_df = pd.DataFrame(predicted_matrix, index=cell_ids, columns=gene_ids)
predicted_df.to_csv("./outputs/hek293t_scribo_predictions.csv")


In [ ]:
exp_values.to_csv("./outputs/hek293t_scribo_expression.csv")


In [ ]:
from torch_geometric.data import DataLoader
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

model_instance= NeuralNet().to(device)
# 创建DataLoader实例
train_loader = DataLoader([train_data], batch_size=1, shuffle=False)

def extract_embeddings(model, loader):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)  # 确保这里data是单个Data对象
            embedding = model.encoder(data.seq)  # 提取encoder部分的输出作为嵌入
            embeddings.append(embedding.cpu().numpy())
    return np.vstack(embeddings)

# 使用DataLoader提取嵌入
embeddings = extract_embeddings(model_instance, train_loader)  # 确保传递的是DataLoader实例

# 计算节点间的相似性
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(embeddings)

# 根据相似性阈值确定新的边
threshold = 0.92  # 设置一个阈值
new_edges = np.where(similarity_matrix > threshold)

# 创建新的PPI网络，此处简单地输出边的列表
new_ppi_edges = list(zip(new_edges[0], new_edges[1]))#

